# 🔥 BlastRadius — A100 Training Notebook
> **Hackathon Day Training Pipeline**  
> Run every cell top-to-bottom. Each stage validates before moving to the next.
>
> **Timeline estimate on A100 80GB (Spot/Preemptible):**
> - Cell 2: SFT data generation ~30-45 min (100 episodes)
> - Cell 3: SFT training ~15-20 min
> - Cell 5: GRPO training ~2-4 hours (WandB tracked, Spot-safe)
> - **Total: ~3-5 hours**

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Environment Setup
# ─────────────────────────────────────────────────────────────
import os

# Verify GPU
!nvidia-smi

# Clone the repo (update URL to your HF Space or GitHub repo)
REPO_URL = "https://github.com/Divyansh-9/BlastRadius.git"  # ← Fixed: use GitHub
!git clone {REPO_URL} blastradius
%cd blastradius

# Install dependencies
!pip install -e '.[train]' -q
# Unsloth (pinned for GRPO compatibility)
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q
!pip install trl>=0.9.0 -q

# Create output dirs
!mkdir -p sft_data models

print('\n✅ Setup complete. GPU ready for training.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — SFT Data Generation via Teacher Model
# ~30-45 min | Uses GPT-4o-mini as teacher for 100 episodes
# ─────────────────────────────────────────────────────────────

# ⚠️  SET YOUR TEACHER API KEY HERE
os.environ['TEACHER_API_KEY'] = 'sk-...'      # ← Your OpenAI / Gemini API key
os.environ['TEACHER_API_BASE'] = 'https://api.openai.com/v1'
os.environ['TEACHER_MODEL'] = 'gpt-4o-mini'   # Cheap and fast enough for SFT data

# Generate 100 episodes across all 3 difficulty tiers
# 100 eps × ~10 steps × 2 roles = ~2000 training examples
!python -m agent.generate_sft_data \
    --episodes 100 \
    --tasks easy medium hard \
    --output sft_data

# Quick sanity check
!echo '--- Line count in expert_trajectories.jsonl ---'
!wc -l sft_data/expert_trajectories.jsonl
print('\n✅ SFT data generation complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Stage 1: Cold-Start SFT Training
# ~15-20 min on A100 | Teaches format + domain vocab to 7B model
# ─────────────────────────────────────────────────────────────
!python -m agent.train_sft \
    --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'  # 7B 4-bit: ~4.5GB, fits A100 safely \
    --data sft_data/expert_trajectories.jsonl \
    --output models/sft_checkpoint

print('\n✅ SFT training complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Validate SFT Save (Critical: §16 Anti-Corruption Check)
# ─────────────────────────────────────────────────────────────
!python -m agent.validate_save --model models/sft_checkpoint

# ⛔ If this cell fails, DO NOT proceed to GRPO.
# Re-run Cell 3 or check disk space.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Stage 3: GRPO Reinforcement Learning (MLOps Optimized)
# SPOT INSTANCE SAFE: Uses WandB for tracking and Async Checkpointing.
# If this job is preempted, it saves an emergency checkpoint automatically.
# ─────────────────────────────────────────────────────────────
import os
os.environ['WANDB_API_KEY'] = os.environ.get('WANDB_API_KEY', '')  # Set WANDB_API_KEY secret in HF Job

!python -m agent.train_grpo \
    --model models/sft_checkpoint \
    --data sft_data/expert_trajectories.jsonl \
    --output models/grpo_checkpoint \
    --hardware-profile a100 \
    --wandb-entity ${WANDB_ENTITY:-blastradius} \
    --hub-model-id ${HUB_MODEL_ID:-Divyansh-9/BlastRadius-GRPO-Checkpoints}

# Expected training log columns to watch:
#   reward/format_reward_func          → should trend ↑ toward 0.75+
#   reward/environment_reward_func     → key metric, watch for positive trend
#   reward                             → overall, watch for upward trend

print('\n✅ GRPO training complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Validate GRPO Save
# ─────────────────────────────────────────────────────────────
!python -m agent.validate_save --model models/grpo_checkpoint

# ⛔ If this cell fails, use models/sft_checkpoint for demo instead.
# A working SFT model is better than a corrupt GRPO model.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Push Trained Model to HF Hub
# ─────────────────────────────────────────────────────────────
from huggingface_hub import HfApi
import os

HF_TOKEN = os.environ.get('HF_TOKEN', '')  # ← Set your HF write token
HF_REPO = os.environ.get('HF_REPO', 'Divyansh-9/BlastRadius-GRPO')

api = HfApi()
api.upload_folder(
    folder_path='models/grpo_checkpoint',
    repo_id=HF_REPO,
    repo_type='model',
    token=HF_TOKEN,
)

print(f'\n✅ Model pushed to https://huggingface.co/{HF_REPO}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — Quick Benchmark: Baseline vs Trained
# Runs 1 episode with zero-shot Qwen and 1 with trained model
# to generate the before/after numbers for the demo
# ─────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')

from incident_env.server.incident_environment import IncidentEnvironment

def score_random_policy(task_id='easy', steps=5):
    """Baseline: random valid commands, no model."""
    import random
    from incident_env.models import VALID_COMMANDS, IncidentAction
    env = IncidentEnvironment()
    env.reset(task_id=task_id)
    total = 0.0
    for _ in range(steps):
        cmd = random.choice(list(VALID_COMMANDS))
        result = env.step(IncidentAction(command=cmd))
        total += result['reward']
        if result['done']:
            break
    return total

# Run 3 baseline episodes
baseline_scores = [score_random_policy('easy') for _ in range(3)]
print(f'Baseline (random policy) mean reward: {sum(baseline_scores)/len(baseline_scores):.4f}')
print('(Compare this with trained model reward from the War Room UI)')